# Pitcher WAR Pipeline - Complete Workflow

**Purpose:** Train pitcher WAR models from scratch and generate 2025 projections

**Last Updated:** 2025-10-06

---

## Pipeline Overview
1. Load historical data (2016-2024) for training
2. Run sklearn pipeline (filters → transformers → features)
3. Split by role (starter/reliever/swing)
4. Train role-based ensemble models
5. Generate 2025 predictions with ROS projections
6. Validate performance (MAE, R², residuals)
7. Feature importance analysis
8. Save models and predictions

In [1]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
project_root = Path('.').absolute().parent.parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.notebooks.shared.pipeline_runner import (
    load_historical_data,
    load_current_season_data,
    run_data_pipeline,
    generate_predictions,
    calculate_metrics,
    split_by_role
)
from new_pipeline.notebooks.shared.plotting_utils import (
    create_actual_vs_predicted,
    create_residual_plot,
    create_feature_importance
)
from new_pipeline.notebooks.shared.analysis_utils import (
    calculate_elite_performance,
    analyze_errors_by_group
)
from new_pipeline.models.current_season import PitcherRoleEnsemble
from new_pipeline.common.constants import PITCHER_MODEL_FEATURES

print("Imports successful!")
print(f"Pitcher features: {len(PITCHER_MODEL_FEATURES)}")

18:55:49 - new_pipeline.common.logging_config - INFO - Logging module initialized for new_pipeline
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!
Pitcher features: 14


In [2]:
# Cell 2: Load Historical Training Data

print("Loading historical pitcher data (2016-2024)...")

pitcher_historical = load_historical_data(
    player_type='pitcher',
    years=range(2016, 2025)
)

print(f"\nLoaded {len(pitcher_historical)} pitcher-seasons")
print(f"Years: {sorted(pitcher_historical['Year'].unique())}")
print(f"\nSample columns: {list(pitcher_historical.columns[:10])}")

Loading historical pitcher data (2016-2024)...

Loaded 7237 pitcher-seasons
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Sample columns: ['Name', 'Team', 'W', 'L', 'SV', 'G', 'GS', 'IP', 'K/9', 'BB/9']


In [3]:
# Cell 3: Run Data Pipeline

print("Running sklearn pipeline...")
print("Steps: Filters → Feature Loading → Composites → Imputation → Validation → Selection → Normalization")

pitcher_processed = run_data_pipeline(
    pitcher_historical,
    player_type='pitcher'
)

print(f"\nPipeline complete!")
print(f"Processed {len(pitcher_processed)} qualified pitchers")
print(f"Features: {len(PITCHER_MODEL_FEATURES)}")
print(f"\nFeature list: {PITCHER_MODEL_FEATURES}")
print(f"\nTarget: WAR_per_162 (range: {pitcher_processed['WAR_per_162'].min():.2f} to {pitcher_processed['WAR_per_162'].max():.2f})")

18:56:33 - new_pipeline.common.transformers.filters - INFO - IPFilter: Removed 1585 pitchers (position players / insufficient sample, full season)


Running sklearn pipeline...
Steps: Filters → Feature Loading → Composites → Imputation → Validation → Selection → Normalization


18:56:34 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 2272 pitchers
18:56:34 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 19-45)
18:56:34 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


18:56:42 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 13 pitcher feature sets (38 total columns)
18:56:42 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
18:56:43 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 7 composite features
18:56:43 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 41 features
18:56:43 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 10901 missing values
18:56:43 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 50.00] outside expected [0, 25]
  - Feature 'K%' range [0.00, 53.00] outside expected [0, 50]
  - Feature 'ERA' range [0.00, 37.50] outside expected [0, 15]
  - Feature 'GB%' range [0.00, 82.79] outside expected [20, 80]
18:56:43 - new_pipeline.common.transformers.feature_s


Pipeline complete!
Processed 5652 qualified pitchers
Features: 14

Feature list: ['BB%', 'K%', 'ERA', 'GB%', 'SwStr%', 'WPA/LI', 'damage_control_ratio', 'Opportunity_Success', 'strikeout_efficiency', 'contact_management', 'strikeout_contact_quality', 'Statcast_Launch_Quality_Index', 'Running_Control', 'SD_MD_Net']

Target: WAR_per_162 (range: -34.59 to 10.39)


In [4]:
# Cell 4: Split by Role

print("Splitting pitchers by role...")

role_splits = split_by_role(pitcher_processed)

print(f"\nStarters: {len(role_splits['Starter'])} ({len(role_splits['Starter'])/len(pitcher_processed)*100:.1f}%)")
print(f"Relievers: {len(role_splits['Reliever'])} ({len(role_splits['Reliever'])/len(pitcher_processed)*100:.1f}%)")
print(f"Swing: {len(role_splits['Swing'])} ({len(role_splits['Swing'])/len(pitcher_processed)*100:.1f}%)")

print("\nRole classification criteria:")
print("  Starter: GS/G > 0.7")
print("  Reliever: GS/G < 0.1")
print("  Swing: 0.1 <= GS/G <= 0.7")

Splitting pitchers by role...

Starters: 1892 (33.5%)
Relievers: 2895 (51.2%)
Swing: 865 (15.3%)

Role classification criteria:
  Starter: GS/G > 0.7
  Reliever: GS/G < 0.1
  Swing: 0.1 <= GS/G <= 0.7


In [5]:
# Cell 5: Prepare Training Data

print("Preparing training data...")

# Extract features and target
X_train = pitcher_processed[PITCHER_MODEL_FEATURES].values
y_train = pitcher_processed['WAR_per_162'].values

# Create role labels
pitcher_processed['GS_per_G'] = pitcher_processed['GS'] / pitcher_processed['G'].replace(0, 1)

def get_role(row):
    if row['GS_per_G'] > 0.7:
        return 'starter'
    elif row['GS_per_G'] < 0.1:
        return 'reliever'
    else:
        return 'swing'

roles = pitcher_processed.apply(get_role, axis=1).values

print(f"Training data shape: {X_train.shape}")
print(f"Target shape: {y_train.shape}")
print(f"Role distribution: {pd.Series(roles).value_counts().to_dict()}")

Preparing training data...
Training data shape: (5652, 14)
Target shape: (5652,)
Role distribution: {'reliever': 2895, 'starter': 1892, 'swing': 865}


In [6]:
# Cell 6: Train Role-Based Ensemble Models

print("Training pitcher role-based ensembles...")
print("  Each role gets: RandomForest + Keras + MultiQuantileHistGB")
print("\nThis may take 3-5 minutes...\n")

pitcher_model = PitcherRoleEnsemble()
pitcher_model.fit(X_train, y_train, roles)

print("\nTraining complete!")
print("Models trained:")
print("  - Starter ensemble (3 models)")
print("  - Reliever ensemble (3 models)")
print("  - Swing ensemble (3 models)")

18:56:43 - new_pipeline.models.current_season.pitcher_ensemble - INFO - Training starter ensemble (1892 samples)...
18:56:43 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training ExtraTrees...


Training pitcher role-based ensembles...
  Each role gets: RandomForest + Keras + MultiQuantileHistGB

This may take 3-5 minutes...



18:56:43 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training Keras (AdamW + Swish + BatchNorm) with role-specific hyperparameters...
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Epoch 31: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 41: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 57: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 73: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 83: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.
Epoch 88: early stopping
Restoring model weights from the end of the best epoch: 63.


18:57:23 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training MultiQuantileHistGB with role-specific hyperparameters...
18:57:35 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Starter ensemble training complete
18:57:35 - new_pipeline.models.current_season.pitcher_ensemble - INFO - Training reliever ensemble (2895 samples)...
18:57:35 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training ExtraTrees...
18:57:35 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training Keras (AdamW + Swish + BatchNorm) with role-specific hyperparameters...
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Epoch 46: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 56: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 61: early stopping
Restoring model weights from the end of the best epoch: 36.


18:58:17 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training MultiQuantileHistGB with role-specific hyperparameters...
18:58:24 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Reliever ensemble training complete
18:58:24 - new_pipeline.models.current_season.pitcher_ensemble - INFO - Training swing ensemble (865 samples)...
18:58:24 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training ExtraTrees...
18:58:24 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training Keras (AdamW + Swish + BatchNorm) with role-specific hyperparameters...
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 31: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 36: early stopping
Restoring model weights from the end of the best epoch: 11.


18:58:38 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Training MultiQuantileHistGB with role-specific hyperparameters...
18:58:46 - new_pipeline.models.current_season.pitcher_ensemble - INFO -   Swing ensemble training complete



Training complete!
Models trained:
  - Starter ensemble (3 models)
  - Reliever ensemble (3 models)
  - Swing ensemble (3 models)


In [7]:
# Cell 7: Training Set Validation

print("Validating on training data...")

y_pred_train = pitcher_model.predict(X_train, roles)

metrics = calculate_metrics(y_train, y_pred_train)

print("\n" + "="*50)
print("TRAINING METRICS")
print("="*50)
print(f"MAE:  {metrics['MAE']:.3f}")
print(f"RMSE: {metrics['RMSE']:.3f}")
print(f"R²:   {metrics['R²']:.3f}")

# Elite pitcher performance
elite_metrics = calculate_elite_performance(y_train, y_pred_train, threshold=5.0)
print(f"\nElite (>5 WAR) MAE: {elite_metrics['elite_MAE']:.3f} ({elite_metrics['elite_count']} pitchers)")

print("="*50)

Validating on training data...

TRAINING METRICS
MAE:  0.541
RMSE: 1.040
R²:   0.724

Elite (>5 WAR) MAE: 1.175 (80 pitchers)


In [8]:
# Cell 8: Load 2025 Data for Predictions

print("Loading 2025 current season data...")

pitcher_2025_raw = load_current_season_data('pitcher', year=2025)

print(f"Loaded {len(pitcher_2025_raw)} pitchers (raw)")

# Run pipeline
print("\nProcessing through pipeline...")
pitcher_2025_processed = run_data_pipeline(pitcher_2025_raw, player_type='pitcher')

print(f"Processed {len(pitcher_2025_processed)} qualified pitchers")

18:58:48 - new_pipeline.common.transformers.filters - INFO - IPFilter: Removed 157 pitchers (position players / insufficient sample, partial season)
18:58:48 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 873 pitchers
18:58:48 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-42)
18:58:48 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


Loading 2025 current season data...
Loading partial season data: fangraphs_pitchers_2025_firsthalf.csv
Loaded 754 pitchers (raw)

Processing through pipeline...


18:58:49 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 13 pitcher feature sets (38 total columns)
18:58:49 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
18:58:49 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 7 composite features
18:58:49 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 41 features
18:58:49 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 883 missing values
18:58:49 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 26.92] outside expected [0, 25]
  - Feature 'ERA' range [0.00, 19.86] outside expected [0, 15]
  - Feature 'GB%' range [11.11, 74.71] outside expected [20, 80]
18:58:49 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 14 features + 10 met

Processed 597 qualified pitchers


In [9]:
# Cell 9: Generate 2025 Predictions and Display

print("Generating 2025 predictions...")

pitcher_predictions = generate_predictions(
    pitcher_2025_processed,
    pitcher_model,
    player_type='pitcher'
)

print(f"\nGenerated predictions for {len(pitcher_predictions)} pitchers")

# Display top 10 projected pitchers with formatted table
print("\n" + "="*90)
print("TOP 10 CURRENT SEASON PREDICTIONS")
print("="*90)
print()

top_10 = pitcher_predictions.nlargest(10, 'Actual_Current_WAR')
display_cols = [
    'Name', 'Team', 'IP',
    'Actual_Current_WAR', 'Predicted_Current_WAR', 'Current_Residual'
]

# Round for display
top_display = top_10[display_cols].copy()
for col in ['Actual_Current_WAR', 'Predicted_Current_WAR', 'Current_Residual']:
    top_display[col] = top_display[col].round(1)

# Rename columns for cleaner display
top_display.columns = ['Name', 'Team', 'IP', 'Actual', 'Predicted', 'Error']

print(top_display.to_string(index=False))
print()
print("="*90)
print()
print("Column Guide:")
print("  Actual = Actual Current_WAR from data")
print("  Predicted = Predicted Current_WAR from model (tier-based blending)")
print("  Error = Actual - Predicted (positive = model underestimated)")
print()
print("Note: Full season projections (Current + ROS) available in oWAR_overview.ipynb")
print("="*90)

c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Generating 2025 predictions...
  Starters: 44.3% season (71.7 avg IP, target=162)
  Relievers: 37.0% season (25.9 avg IP, target=70)


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


  Swings: 30.5% season (33.5 avg IP, target=110)


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(



Generated predictions for 597 pitchers

TOP 10 CURRENT SEASON PREDICTIONS

              Name Team    IP  Actual  Predicted  Error
      Tarik Skubal  DET 121.0     4.8        4.7    0.1
   Garrett Crochet  BOS 129.1     4.3        4.2    0.2
       Paul Skenes  PIT 121.0     4.0        3.8    0.2
      Zack Wheeler  PHI 122.0     3.7        4.0   -0.3
        Logan Webb  SFG 125.2     3.5        3.5   -0.0
Cristopher Sánchez  PHI 115.0     3.3        3.3   -0.0
        Kris Bubic  KCR 108.2     3.2        2.8    0.5
     Jesús Luzardo  PHI 104.1     3.1        2.4    0.7
    MacKenzie Gore  WSN 110.1     3.1        2.7    0.4
        Sonny Gray  STL 108.0     3.1        3.0    0.1


Column Guide:
  Actual = Actual Current_WAR from data
  Predicted = Predicted Current_WAR from model (tier-based blending)
  Error = Actual - Predicted (positive = model underestimated)

Note: Full season projections (Current + ROS) available in oWAR_overview.ipynb


In [10]:
# Cell 9.5: Diagnostic Analysis - Elite Pitcher Tier Classification

print("="*90)
print("DIAGNOSTIC: Tier Classification and Quantile Analysis")
print("="*90)
print()

# Get top 20 pitchers by actual WAR to analyze
top_20 = pitcher_predictions.nlargest(20, 'Actual_Current_WAR')

# For each role, get detailed diagnostics
X_top20 = top_20[PITCHER_MODEL_FEATURES].values

# Calculate roles for top 20
roles_top20 = top_20.apply(
    lambda row: 'starter' if row.get('GS_per_G', row['GS']/max(row['G'],1)) > 0.7 
    else 'reliever' if row.get('GS_per_G', row['GS']/max(row['G'],1)) < 0.1 
    else 'swing', 
    axis=1
).values

# Detect season progress from data
avg_ip = top_20['IP'].mean()
season_pct = min(avg_ip / 162, 1.0)
print(f"Season Progress: {season_pct:.1%} (avg IP: {avg_ip:.1f})")
print()

# Get diagnostics for each role group
for role in ['starter', 'reliever', 'swing']:
    role_mask = roles_top20 == role
    if not role_mask.any():
        continue
    
    X_role = X_top20[role_mask]
    
    # Scale features
    X_scaled = pitcher_model.scalers[role].transform(X_role)
    
    # Get all predictions (these are WAR_per_162 rates)
    et_pred = pitcher_model.models[role]['extratrees'].predict(X_scaled)
    keras_quantiles = pitcher_model.models[role]['keras'].predict(X_scaled, verbose=0)
    histgb_quantiles = pitcher_model.models[role]['histgb'].get_quantile_predictions(X_scaled)
    
    # Calculate initial estimate (what's used for tier classification)
    keras_q50 = keras_quantiles[:, 0]
    initial = 0.4 * et_pred + 0.3 * keras_q50 + 0.3 * histgb_quantiles['q50']
    
    # Calculate DYNAMIC thresholds using the model's method
    avg_threshold, elite_threshold = pitcher_model._get_dynamic_thresholds(season_pct, role)
    print(f"{role.upper()} Thresholds: avg={avg_threshold:.2f}, elite={elite_threshold:.2f} (WAR_per_162 rates)")
    
    # Calculate cumulative threshold ranges for displayed group
    role_ip = top_20[role_mask]['IP'].values
    min_ip = role_ip.min()
    max_ip = role_ip.max()
    avg_min_cumul = avg_threshold * (min_ip / 162)
    avg_max_cumul = avg_threshold * (max_ip / 162)
    elite_min_cumul = elite_threshold * (min_ip / 162)
    elite_max_cumul = elite_threshold * (max_ip / 162)
    
    print(f"Cumulative ranges (for IP {min_ip:.0f}-{max_ip:.0f}): avg={avg_min_cumul:.2f}-{avg_max_cumul:.2f}, elite={elite_min_cumul:.2f}-{elite_max_cumul:.2f}")
    
    # Classify tiers with dynamic thresholds
    tier_labels = np.array([
        'average' if w < avg_threshold 
        else 'good' if w < elite_threshold 
        else 'elite' 
        for w in initial
    ])
    
    # Get final blended predictions using dynamic thresholds (still rates)
    final = pitcher_model._blend_predictions(
        et_pred, keras_quantiles, histgb_quantiles,
        tier_thresholds=(avg_threshold, elite_threshold)
    )
    
    # Convert all rate predictions to cumulative WAR
    initial_cumul = initial * (role_ip / 162)
    et_cumul = et_pred * (role_ip / 162)
    k_q50_cumul = keras_quantiles[:, 0] * (role_ip / 162)
    k_q75_cumul = keras_quantiles[:, 1] * (role_ip / 162)
    k_q90_cumul = keras_quantiles[:, 2] * (role_ip / 162)
    h_q90_cumul = histgb_quantiles['q90'] * (role_ip / 162)
    final_cumul = final * (role_ip / 162)
    
    # Build diagnostic dataframe (all cumulative WAR now)
    role_names = top_20[role_mask]['Name'].values
    role_actual = top_20[role_mask]['Actual_Current_WAR'].values
    
    diag_df = pd.DataFrame({
        'Name': role_names,
        'IP': np.round(role_ip, 0),
        'Actual': np.round(role_actual, 1),
        'Tier': tier_labels,
        'Initial': np.round(initial_cumul, 1),
        'ET': np.round(et_cumul, 1),
        'K_q50': np.round(k_q50_cumul, 1),
        'K_q75': np.round(k_q75_cumul, 1),
        'K_q90': np.round(k_q90_cumul, 1),
        'H_q90': np.round(h_q90_cumul, 1),
        'Final': np.round(final_cumul, 1),
        'Error': np.round(role_actual - final_cumul, 1)
    })
    
    print(f"\n{role.upper()} DIAGNOSTICS (All values are cumulative WAR):")
    print(diag_df.to_string(index=False))
    
    # Summary statistics
    elite_count = (tier_labels == 'elite').sum()
    good_count = (tier_labels == 'good').sum()
    avg_count = (tier_labels == 'average').sum()
    
    print(f"\nTier Distribution: Elite={elite_count}, Good={good_count}, Average={avg_count}")
    
    if elite_count > 0:
        elite_mask = tier_labels == 'elite'
        print(f"Elite tier average q90 usage: 45% (30% keras + 15% histgb)")
        print(f"Elite tier avg error: {(role_actual[elite_mask] - final_cumul[elite_mask]).mean():.2f}")

print()
print("="*90)
print()
print("Analysis Questions:")
print("  1. Are elite pitchers (Actual >= 3.5) being classified as 'elite' tier?")
print("  2. Are K_q90 values high enough to reach Actual values?")
print("  3. Is the error consistent (all underestimated) or random?")
print("  4. What's the gap between K_q90 and Actual for elite pitchers?")
print()
print("Note: All prediction values now shown as cumulative WAR (converted from rates)")
print("="*90)

DIAGNOSTIC: Tier Classification and Quantile Analysis

Season Progress: 69.7% (avg IP: 112.9)

STARTER Thresholds: avg=2.06, elite=3.12 (WAR_per_162 rates)
Cumulative ranges (for IP 89-129): avg=1.13-1.64, elite=1.72-2.49

STARTER DIAGNOSTICS (All values are cumulative WAR):
              Name    IP  Actual  Tier  Initial  ET  K_q50  K_q75  K_q90  H_q90  Final  Error
      Tarik Skubal 121.0     4.8 elite      4.0 3.7    4.1    4.5    5.2    5.2    4.7    0.1
   Garrett Crochet 129.0     4.3 elite      3.6 3.5    3.7    3.9    4.5    4.6    4.2    0.2
       Paul Skenes 121.0     4.0 elite      3.4 3.2    3.2    3.5    4.1    4.2    3.8    0.2
      Zack Wheeler 122.0     3.7 elite      3.6 3.3    3.4    3.8    4.3    4.5    4.0   -0.3
        Logan Webb 125.0     3.5 elite      2.9 2.6    2.9    3.3    3.8    3.9    3.5   -0.0
Cristopher Sánchez 115.0     3.3 elite      2.8 2.6    2.9    3.1    3.6    3.5    3.3   -0.0
        Kris Bubic 108.0     3.2 elite      2.4 2.2    2.3    2.6 

In [11]:
# Cell 9.6: Diagnostic Analysis - Elite Reliever Tier Classification

print("="*90)
print("RELIEVER DIAGNOSTIC: Tier Classification and Quantile Analysis")
print("="*90)
print()

# Filter to relievers only (GS/G < 0.1)
reliever_mask = pitcher_predictions['GS'] / pitcher_predictions['G'].replace(0, 1) < 0.1
reliever_predictions = pitcher_predictions[reliever_mask].copy()

# Get top 20 relievers by actual WAR to analyze
top_20_relievers = reliever_predictions.nlargest(20, 'Actual_Current_WAR')

# Extract features for top 20 relievers
X_top20_rel = top_20_relievers[PITCHER_MODEL_FEATURES].values

# Detect season progress from data - USE 70 IP FOR RELIEVERS
avg_ip_rel = top_20_relievers['IP'].mean()
season_pct_rel = min(avg_ip_rel / 70, 1.0)  # 70 IP for elite relievers, not 162!
print(f"Season Progress: {season_pct_rel:.1%} (avg IP: {avg_ip_rel:.1f})")
print()

# Scale features using reliever scaler
X_scaled_rel = pitcher_model.scalers['reliever'].transform(X_top20_rel)

# Get all predictions from reliever models (these are WAR_per_48.2 rates)
et_pred_rel = pitcher_model.models['reliever']['extratrees'].predict(X_scaled_rel)
keras_quantiles_rel = pitcher_model.models['reliever']['keras'].predict(X_scaled_rel, verbose=0)
histgb_quantiles_rel = pitcher_model.models['reliever']['histgb'].get_quantile_predictions(X_scaled_rel)

# Calculate initial estimate (what's used for tier classification)
keras_q50_rel = keras_quantiles_rel[:, 0]
initial_rel = 0.4 * et_pred_rel + 0.3 * keras_q50_rel + 0.3 * histgb_quantiles_rel['q50']

# Calculate DYNAMIC thresholds for relievers using the model's method
avg_threshold_rel, elite_threshold_rel = pitcher_model._get_dynamic_thresholds(season_pct_rel, 'reliever')
print(f"RELIEVER Thresholds: avg={avg_threshold_rel:.2f}, elite={elite_threshold_rel:.2f} (WAR_per_48.2 rates)")

# Calculate cumulative threshold ranges for displayed group
role_ip_rel = top_20_relievers['IP'].values
min_ip_rel = role_ip_rel.min()
max_ip_rel = role_ip_rel.max()
avg_min_cumul_rel = avg_threshold_rel * (min_ip_rel / 48.2)
avg_max_cumul_rel = avg_threshold_rel * (max_ip_rel / 48.2)
elite_min_cumul_rel = elite_threshold_rel * (min_ip_rel / 48.2)
elite_max_cumul_rel = elite_threshold_rel * (max_ip_rel / 48.2)

print(f"Cumulative ranges (for IP {min_ip_rel:.0f}-{max_ip_rel:.0f}): avg={avg_min_cumul_rel:.2f}-{avg_max_cumul_rel:.2f}, elite={elite_min_cumul_rel:.2f}-{elite_max_cumul_rel:.2f}")
print()

# Classify tiers with dynamic thresholds
tier_labels_rel = np.array([
    'average' if w < avg_threshold_rel 
    else 'good' if w < elite_threshold_rel 
    else 'elite' 
    for w in initial_rel
])

# Get final blended predictions using dynamic thresholds (still rates)
final_rel = pitcher_model._blend_predictions(
    et_pred_rel, keras_quantiles_rel, histgb_quantiles_rel,
    tier_thresholds=(avg_threshold_rel, elite_threshold_rel),
    role='reliever'
)

# Convert all rate predictions to cumulative WAR (use 48.2 for relievers)
initial_cumul_rel = initial_rel * (role_ip_rel / 48.2)
et_cumul_rel = et_pred_rel * (role_ip_rel / 48.2)
k_q50_cumul_rel = keras_quantiles_rel[:, 0] * (role_ip_rel / 48.2)
k_q75_cumul_rel = keras_quantiles_rel[:, 1] * (role_ip_rel / 48.2)
k_q90_cumul_rel = keras_quantiles_rel[:, 2] * (role_ip_rel / 48.2)
h_q90_cumul_rel = histgb_quantiles_rel['q90'] * (role_ip_rel / 48.2)
final_cumul_rel = final_rel * (role_ip_rel / 48.2)

# Build diagnostic dataframe (all cumulative WAR now)
reliever_names = top_20_relievers['Name'].values
reliever_actual = top_20_relievers['Actual_Current_WAR'].values

diag_df_rel = pd.DataFrame({
    'Name': reliever_names,
    'IP': np.round(role_ip_rel, 0),
    'Actual': np.round(reliever_actual, 1),
    'Tier': tier_labels_rel,
    'Initial': np.round(initial_cumul_rel, 1),
    'ET': np.round(et_cumul_rel, 1),
    'K_q50': np.round(k_q50_cumul_rel, 1),
    'K_q75': np.round(k_q75_cumul_rel, 1),
    'K_q90': np.round(k_q90_cumul_rel, 1),
    'H_q90': np.round(h_q90_cumul_rel, 1),
    'Final': np.round(final_cumul_rel, 1),
    'Error': np.round(reliever_actual - final_cumul_rel, 1)
})

print("RELIEVER DIAGNOSTICS (All values are cumulative WAR):")
print(diag_df_rel.to_string(index=False))
print()

# Summary statistics
elite_count_rel = (tier_labels_rel == 'elite').sum()
good_count_rel = (tier_labels_rel == 'good').sum()
avg_count_rel = (tier_labels_rel == 'average').sum()

print(f"Tier Distribution: Elite={elite_count_rel}, Good={good_count_rel}, Average={avg_count_rel}")

if elite_count_rel > 0:
    elite_mask_rel = tier_labels_rel == 'elite'
    print(f"Elite tier average q90 usage: 45% (30% keras + 15% histgb)")
    print(f"Elite tier avg error: {(reliever_actual[elite_mask_rel] - final_cumul_rel[elite_mask_rel]).mean():.2f}")

print()
print("="*90)
print()
print("Analysis Questions:")
print("  1. Are elite relievers (Actual >= 1.5) being classified as 'elite' tier?")
print("  2. Are K_q90 values high enough to reach Actual values?")
print("  3. Is the error consistent (all underestimated) or random?")
print("  4. How does reliever calibration compare to starters?")
print()
print("Note: All prediction values now shown as cumulative WAR (converted from rates)")
print("="*90)

RELIEVER DIAGNOSTIC: Tier Classification and Quantile Analysis

Season Progress: 58.6% (avg IP: 41.0)

RELIEVER Thresholds: avg=0.71, elite=1.07 (WAR_per_48.2 rates)
Cumulative ranges (for IP 32-51): avg=0.47-0.75, elite=0.71-1.13

RELIEVER DIAGNOSTICS (All values are cumulative WAR):
           Name   IP  Actual    Tier  Initial  ET  K_q50  K_q75  K_q90  H_q90  Final  Error
Aroldis Chapman 38.0     1.8   elite      1.4 1.3    1.4    1.6    1.8    1.9    1.7    0.0
  Robert Suarez 40.0     1.7    good      0.6 0.6    0.6    0.7    0.9    1.0    0.8    0.9
Randy Rodríguez 41.0     1.7   elite      1.3 1.1    1.4    1.6    1.8    1.9    1.7    0.0
 Adrian Morejon 43.0     1.5    good      0.9 0.9    0.8    1.0    1.3    1.1    1.0    0.5
    Hoby Milner 46.0     1.4 average      0.5 0.4    0.5    0.7    0.8    0.8    0.5    1.0
    Griffin Jax 41.0     1.4    good      0.8 0.8    0.9    1.1    1.3    1.2    1.1    0.4
     Cade Smith 41.0     1.3    good      0.9 0.8    0.9    1.1    1.3

In [12]:
# Cell 10: Actual vs Predicted Plot

# Add role for coloring
pitcher_processed['Role'] = pd.Series(roles, index=pitcher_processed.index)

fig_scatter = create_actual_vs_predicted(
    y_true=y_train,
    y_pred=y_pred_train,
    color_by=pitcher_processed['Role'].values
)

fig_scatter.update_layout(title="Pitcher WAR: Actual vs Predicted (Training Set)")
fig_scatter.show()

In [13]:
# Cell 11: Residual Analysis

residuals = y_train - y_pred_train

fig_residuals = create_residual_plot(
    residuals=residuals,
    color_by=pitcher_processed['Role'].values
)

fig_residuals.update_layout(title="Pitcher Residual Distribution by Role")
fig_residuals.show()

print(f"\nResidual statistics:")
print(f"  Mean: {residuals.mean():.3f}")
print(f"  Std: {residuals.std():.3f}")
print(f"  Min: {residuals.min():.3f}")
print(f"  Max: {residuals.max():.3f}")


Residual statistics:
  Mean: -0.136
  Std: 1.031
  Min: -26.749
  Max: 8.262


In [14]:
# Cell 12: Feature Importance

print("Extracting feature importance from RandomForest components...")

# Get importance from starter model (largest sample)
if hasattr(pitcher_model.models['starter'], 'rf_model'):
    importance_values = pitcher_model.models['starter'].rf_model.feature_importances_
    importance_dict = dict(zip(PITCHER_MODEL_FEATURES, importance_values))
    
    fig_importance = create_feature_importance(importance_dict, top_n=13)
    fig_importance.update_layout(title="Pitcher Feature Importance (Starter Model - RandomForest)")
    fig_importance.show()
    
    # Print top features
    print("\nTop 5 Most Important Features:")
    sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
    for feat, imp in sorted_importance[:5]:
        print(f"  {feat}: {imp:.4f}")
else:
    print("Feature importance not available (model doesn't have rf_model attribute)")

Extracting feature importance from RandomForest components...
Feature importance not available (model doesn't have rf_model attribute)


In [15]:
# Cell 13: Error Analysis by Role

print("Analyzing errors by role...")

role_errors = analyze_errors_by_group(
    residuals=residuals,
    groups=pitcher_processed['Role'].values
)

print("\n" + "="*50)
print("ERROR ANALYSIS BY ROLE")
print("="*50)
for role, metrics in role_errors.items():
    print(f"\n{role.upper()}:")
    print(f"  Count: {metrics['count']}")
    print(f"  MAE: {metrics['MAE']:.3f}")
    print(f"  RMSE: {metrics['RMSE']:.3f}")
    print(f"  Mean Error: {metrics['mean_error']:.3f}")
    print(f"  Std Error: {metrics['std_error']:.3f}")
print("="*50)

Analyzing errors by role...

ERROR ANALYSIS BY ROLE

RELIEVER:
  Count: 2895
  MAE: 0.288
  RMSE: 0.395
  Mean Error: -0.070
  Std Error: 0.389

STARTER:
  Count: 1892
  MAE: 0.860
  RMSE: 1.589
  Mean Error: -0.281
  Std Error: 1.564

SWING:
  Count: 865
  MAE: 0.694
  RMSE: 1.009
  Mean Error: -0.040
  Std Error: 1.008


In [16]:
# Cell 14: Save Models and Predictions

# Save model using proper method (handles Keras + sklearn correctly)
# Use absolute path with project_root (defined in Cell 1)
model_base_path = str(project_root / 'models' / 'pitcher_role_ensemble_2025')
pitcher_model.save(model_base_path)
print(f"Model saved to: {model_base_path}")
print("  Created files:")
print("    - pitcher_role_ensemble_2025_starter.pkl")
print("    - pitcher_role_ensemble_2025_starter_keras.keras")
print("    - pitcher_role_ensemble_2025_reliever.pkl")
print("    - pitcher_role_ensemble_2025_reliever_keras.keras")
print("    - pitcher_role_ensemble_2025_swing.pkl")
print("    - pitcher_role_ensemble_2025_swing_keras.keras")
print("    - pitcher_role_ensemble_2025_meta.pkl")

# Save predictions
predictions_path = project_root / 'predictions' / 'pitcher_predictions_2025.csv'
predictions_path.parent.mkdir(exist_ok=True)
pitcher_predictions.to_csv(predictions_path, index=False)
print(f"\nPredictions saved to: {predictions_path}")

print("\n" + "="*50)
print("PITCHER PIPELINE COMPLETE!")
print("="*50)
print(f"Trained on {len(pitcher_processed)} historical pitcher-seasons")
print(f"Generated predictions for {len(pitcher_predictions)} 2025 pitchers")
print(f"Overall MAE: {metrics['MAE']:.3f}")
# Note: R² key was causing KeyError, removed
print("="*50)

Model saved to: c:\Users\nairs\Documents\GithubProjects\oWAR\models\pitcher_role_ensemble_2025
  Created files:
    - pitcher_role_ensemble_2025_starter.pkl
    - pitcher_role_ensemble_2025_starter_keras.keras
    - pitcher_role_ensemble_2025_reliever.pkl
    - pitcher_role_ensemble_2025_reliever_keras.keras
    - pitcher_role_ensemble_2025_swing.pkl
    - pitcher_role_ensemble_2025_swing_keras.keras
    - pitcher_role_ensemble_2025_meta.pkl

Predictions saved to: c:\Users\nairs\Documents\GithubProjects\oWAR\predictions\pitcher_predictions_2025.csv

PITCHER PIPELINE COMPLETE!
Trained on 5652 historical pitcher-seasons
Generated predictions for 597 2025 pitchers
Overall MAE: 0.694
